# 0. Getting Started (Fraud Prevention Challenge)

**Mercado Libre: Data Scientist Technical Challenge**

Mercado Libre procesa miles de transacciones por segundo, lo que la convierte en blanco frecuente de ataques de fraude en pagos. El objetivo de este proyecto es construir un modelo de Machine Learning que prediga si una transacción es fraudulenta, maximizando la **ganancia económica de la empresa** - no solo métricas de clasificación estándar (accuracy, F1, etc.).

Este notebook introductorio define:
1. La **ecuación de beneficio** que traduce la regla de negocio en una función objetivo optimizable.
2. El **plan de trabajo** que se seguirá en los notebooks siguientes para resolver el challenge.

## 1. Ecuación de beneficio (aceptar vs. rechazar)

### Regla de negocio

Según el enunciado del challenge:

- Si se **aprueba** una transacción **legítima**, la empresa gana el **25%** del monto de la transacción.
- Si se **aprueba** una transacción **fraudulenta**, la empresa pierde el **100%** del monto (el dinero no se recupera).
- Si se **rechaza** una transacción (legítima o fraudulenta), no hay movimiento de dinero: no se gana comisión, pero tampoco se pierde capital.

### Matriz de beneficio por transacción

| Decisión / Realidad | Legítima (y=0) | Fraude (y=1) |
|---|---|---|
| **Aceptar** | `+0.25 * monto` | `-1.00 * monto` |
| **Rechazar** | `0` | `0` |

### Beneficio esperado de aceptar una transacción

Sea `p` la probabilidad estimada de que una transacción sea fraude (dada por el modelo) y `monto` su valor. El beneficio esperado de **aceptarla** es:

```
E[beneficio | aceptar] = 0.25 * monto * (1 - p) - 1.00 * monto * p
```

El beneficio esperado de **rechazar** es siempre `0`.

### Umbral de decisión óptimo

Conviene aceptar la transacción solo si su beneficio esperado supera al de rechazar (que es 0):

```
0.25 * (1 - p) - 1.00 * p > 0   =>   p < 0.25 / (0.25 + 1.00) = 0.20
```

**Conclusión clave:** el `monto` se cancela en la desigualdad, por lo que — bajo esta regla lineal de ganancia/pérdida - el **umbral óptimo de probabilidad de fraude es 20%**, independiente del monto de la transacción. El monto sí determina la *magnitud* de la ganancia o pérdida, pero no dónde conviene ubicar el punto de corte de decisión.

En términos generales, para una tasa de ganancia `g` y una tasa de pérdida `l`, el umbral óptimo es `p* = g / (g + l)`.

### Beneficio total del portafolio

La métrica de negocio a **maximizar** en todo el proyecto es la suma de los beneficios individuales sobre todas las transacciones aceptadas:

```
Beneficio_total = Σ (sobre transacciones aceptadas) [ 0.25 * monto_i * (1 - y_i) - 1.00 * monto_i * y_i ]
```

donde `y_i = 1` si la transacción `i` es fraude real y `0` si es legítima. Esta será la métrica principal de evaluación de los modelos, comparada contra los baselines triviales de *aprobar todo* y *rechazar todo*.

In [3]:
def transaction_profit(is_fraud: int, approved: bool, amount: float,
                        gain_rate: float = 0.25, loss_rate: float = 1.0) -> float:
    """Beneficio de una transacción individual según la decisión tomada."""
    if not approved:
        return 0.0
    return -loss_rate * amount if is_fraud else gain_rate * amount


def portfolio_profit(y_true, approved, amount,
                      gain_rate: float = 0.25, loss_rate: float = 1.0) -> float:
    """Beneficio total de un conjunto de transacciones."""
    return sum(
        transaction_profit(y, a, m, gain_rate, loss_rate)
        for y, a, m in zip(y_true, approved, amount)
    )


def optimal_threshold(gain_rate: float = 0.25, loss_rate: float = 1.0) -> float:
    """Umbral óptimo de probabilidad de fraude por debajo del cual conviene aceptar."""
    return gain_rate / (gain_rate + loss_rate)

In [4]:
assert transaction_profit(is_fraud=0, approved=True, amount=100) == 25.0
assert transaction_profit(is_fraud=1, approved=True, amount=100) == -100.0
assert transaction_profit(is_fraud=0, approved=False, amount=100) == 0.0
assert transaction_profit(is_fraud=1, approved=False, amount=100) == 0.0
assert optimal_threshold() == 0.20

print(f"Umbral óptimo de probabilidad de fraude: {optimal_threshold():.2%}")

Umbral óptimo de probabilidad de fraude: 20.00%


## 2. Plan de trabajo

Roadmap para resolver el challenge, alineado a lo solicitado en el enunciado:

1. **EDA** - nulos, distribución de `monto` y `score`, variables `a`..`p`, desbalance de clases, patrones temporales en `fecha`, correlación con `fraude`.
2. **Limpieza y feature engineering** - tratamiento de nulos y categóricas, features derivadas de `fecha`, análisis del `score` preexistente.
3. **Split de datos** - split temporal (train/val/test por `fecha`) para simular condiciones de producción y evitar leakage.
4. **Modelado** - baseline (Regresión Logística) vs. modelos más robustos (Random Forest, Gradient Boosting/XGBoost, LightGBM), con manejo de desbalance de clases.
5. **Optimización orientada a negocio** - en vez de umbral 0.5, usar la ecuación de beneficio de la Sección 1 para elegir el umbral (o política) que maximiza el beneficio total en validación, comparado contra los baselines triviales *aprobar todo* / *rechazar todo*.
6. **Evaluación** - métricas de clasificación (ROC-AUC, PR-AUC, F1, F6) + métrica de negocio (beneficio total, beneficio por transacción).
7. **Interpretabilidad y robustez** - importancia de variables, estabilidad del umbral elegido.
8. **Informe** - hipótesis, análisis, modelos, evaluación, conclusión (entregable en PDF).
9. **Preguntas de producción** - reproducibilidad lab→prod, causas de degradación de performance, pasos de despliegue del modelo.